
# 01 Core Python для ML Engineer Interview

Этот ноутбук — структурированный конспект и тренажёр для собеседований на позиции ML Engineer.  
Фокус: поведение объектов Python, производительность, память, алгоритмическая сложность и практические edge-cases.

**Формат каждого раздела:**
1. Теория
2. Код
3. Разбор кода
4. Производительность и память
5. Вопросы с собеседований
6. Мини-задачи
7. Edge cases


In [ ]:
import sys
import timeit
import copy
from collections import deque


## Mutable vs Immutable types

### 1) Теория

В Python изменяемость относится к **состоянию объекта**, а не к имени переменной.  
`a = b` копирует ссылку на объект, а не сам объект.  

- **Immutable**: `int`, `float`, `bool`, `str`, `tuple` (если элементы тоже immutable), `frozenset`, `bytes`.
- **Mutable**: `list`, `dict`, `set`, `bytearray`, большинство пользовательских объектов.

Ключевой эффект: операции над immutable часто создают **новый объект**, а над mutable — изменяют объект **in-place**.


In [ ]:
x = 10
print(id(x))
x += 1
print(id(x), x)  # новый объект int

lst = [1, 2, 3]
print(id(lst))
lst.append(4)
print(id(lst), lst)  # тот же объект list

a = [1, 2]
b = a
b.append(3)
print(a, b, a is b)


### 3) Разбор кода

У `int` после `x += 1` меняется `id`, потому что создаётся новый объект.  
`list.append` меняет исходный список, `id` остаётся прежним.  
`b = a` создаёт вторую ссылку на тот же list: изменение через `b` видно через `a`.


### 4) Производительность и память

- Частая ошибка в проде: мутация объекта, переданного в функцию, вызывает скрытые side-effects.
- Immutable типы безопаснее для кешей и многопоточности (меньше синхронизации).
- Создание новых immutable-объектов (например, частая конкатенация строк через `+`) может быть дорого; используйте `"".join(...)`.


### 5) Вопросы с собеседований

1. Почему `tuple` не всегда “полностью immutable”?
2. Какие баги вызывает mutable default argument?
3. Когда выгоднее возвращать новый объект, а не мутировать входной?


### 6) Практические мини-задачи

1. Напишите функцию, которая безопасно добавляет элемент в список без мутации исходного списка.
2. Продемонстрируйте баг с `def f(x=[])` и исправьте его.


### 7) Edge cases

- `tuple` может содержать mutable объекты, и тогда “глубинное состояние” меняется.
- Интернирование строк/чисел может визуально путать `id` и `is`.


## is vs ==

### 1) Теория

`==` сравнивает **значение** (через `__eq__`), `is` сравнивает **идентичность** (один ли объект в памяти).  

Правило: `is` использовать для `None`, `True`, `False`, singleton-объектов. Для значений — `==`.


In [ ]:
a = [1, 2]
b = [1, 2]
print(a == b, a is b)

c = None
print(c is None, c == None)  # второй стиль не рекомендуется (E711)

x = 256
y = 256
print(x is y)  # может быть True из-за кеширования small integers


### 3) Разбор кода
`a` и `b` равны по содержимому, но это разные объекты. `None` проверяем через `is None`. С integer interning `is` для чисел ненадёжен.

### 4) Производительность и память
`is` — O(1) сравнение указателей; `==` может быть дорогим (например, длинные списки/кастомный `__eq__`). Но выбирать нужно по семантике, а не по микроскорости.

### 5) Вопросы с собеседований

1. Почему `x is 1000` — плохая идея?
2. Когда `__eq__` может быть небезопасен/неоптимален?


### 6) Практические мини-задачи

1. Реализуйте класс с корректным `__eq__` и объясните, как это влияет на сравнение.
2. Найдите в своём коде места, где `== None` и исправьте.


### 7) Edge cases

- Объекты могут переопределять `__eq__` неожиданно (например, NumPy arrays возвращают массив bool).
- `float('nan') == float('nan')` даёт `False`.


## Copy vs deepcopy

### 1) Теория

`copy.copy` делает **поверхностную копию**: копируется внешний контейнер, вложенные объекты остаются общими.  
`copy.deepcopy` рекурсивно копирует весь граф объектов (с memo для циклов).


In [ ]:
original = [[1, 2], [3, 4]]
shallow = copy.copy(original)
deep = copy.deepcopy(original)

shallow[0].append(99)
print("original:", original)
print("shallow:", shallow)
print("deep:", deep)

print(original is shallow, original[0] is shallow[0], original[0] is deep[0])


### 3) Разбор кода
После shallow-copy вложенный список общий, поэтому изменение видно в `original`. `deepcopy` создаёт независимые вложенные структуры.

### 4) Производительность и память
`deepcopy` может быть очень дорогим по CPU/RAM. Для продакшена часто лучше проектировать неизменяемые структуры или явно копировать только изменяемые ветки (copy-on-write подход).

### 5) Вопросы с собеседований

1. Почему `deepcopy` может сломаться/быть медленным на больших графах объектов?
2. Когда достаточно shallow copy?


### 6) Практические мини-задачи

1. Реализуйте функцию обновления конфигурации (dict) без мутации входа.
2. Измерьте разницу времени между `copy` и `deepcopy` на вложенных структурах.


### 7) Edge cases

- Кастомные классы с `__getstate__`, дескрипторами, файловыми хендлами требуют аккуратности.
- Циклические ссылки обрабатываются через memo, но цена копирования растёт.


## Hashability

### 1) Теория

Объект hashable, если имеет стабильный `__hash__` и корректный `__eq__`.  
Hashable объекты можно использовать как ключи словаря и элементы множества.

Контракт: если `a == b`, то `hash(a) == hash(b)`.


In [ ]:
print(hash("ml"), hash((1, 2, 3)))

try:
    hash([1, 2, 3])
except TypeError as e:
    print("Error:", e)

s = {(1, 2), (3, 4)}
print((1, 2) in s)


### 3) Разбор кода
Строка и tuple hashable, список — нет, потому что mutable и его hash мог бы измениться после вставки в dict/set.

### 4) Производительность и память
Коллизии хэшей возможны; lookup в среднем O(1), в худшем O(n). Для пользовательских классов важно корректно реализовать `__eq__`/`__hash__`, иначе деградация и баги.

### 5) Вопросы с собеседований

1. Почему mutable объект как ключ dict опасен?
2. Что произойдёт, если переопределить `__eq__`, но не `__hash__`?


### 6) Практические мини-задачи

1. Создайте dataclass для ключа кеша и сделайте его безопасно hashable.
2. Проверьте, как меняется поведение set при коллизиях.


### 7) Edge cases

- `tuple` hashable только если все его элементы hashable.
- Для `float('nan')` поведение в множествах/словарях может быть контринтуитивным.


## Внутреннее устройство dict

### 1) Теория

`dict` в CPython — хеш-таблица с открытой адресацией.  
Основные этапы: вычисление хэша ключа → индекс в таблице → probe sequence при коллизиях.  

Свойства:
- Амортизированно O(1) для `get/set/del`.
- При росте заполняемости выполняется resize (реаллоцируется таблица большего размера).
- С Python 3.7+ порядок вставки гарантирован как часть языка.


In [ ]:
d = {str(i): i for i in range(10_000)}
print(d["42"], "9999" in d)

def dict_lookup_benchmark(n=100_000):
    d = {i: i for i in range(n)}
    return timeit.timeit('d[50000]', number=1_000_000, globals={'d': d})

print("lookup time:", round(dict_lookup_benchmark(), 4), "sec")


### 3) Разбор кода
Создаётся большой словарь и измеряется скорость повторного доступа к ключу. Это иллюстрирует O(1) в среднем для lookup.

### 4) Производительность и память
При коллизиях и плохих хэшах стоимость растёт. Resize дорогой точечно, но даёт амортизированную эффективность. Для hot-path важно избегать лишних аллокаций и постоянно пересоздаваемых dict.

### 5) Вопросы с собеседований

1. Почему dict операции называют амортизированно O(1), а не строго O(1)?
2. Как хэш-коллизии влияют на latency?
3. Что такое hash flooding и как Python защищается?


### 6) Практические мини-задачи

1. Сымитируйте коллизии кастомным классом с одинаковым `__hash__`.
2. Сравните скорость lookup до и после существенного роста словаря.


### 7) Edge cases

- Мутация объекта-ключа (если обошли ограничения) ломает адресацию.
- Большие dict могут фрагментировать память и ухудшать cache locality.


## list vs tuple vs set (память и производительность)

### 1) Теория

- `list`: динамический массив ссылок, быстрый доступ по индексу O(1), append амортизированно O(1).
- `tuple`: immutable последовательность, обычно компактнее list и чуть быстрее для чтения.
- `set`: хеш-таблица уникальных элементов, membership в среднем O(1).


In [ ]:
data = list(range(10000))
t_data = tuple(data)
s_data = set(data)

print("sizes:", sys.getsizeof(data), sys.getsizeof(t_data), sys.getsizeof(s_data))

print("list membership:", timeit.timeit('9999 in data', number=200000, globals=globals()))
print("tuple membership:", timeit.timeit('9999 in t_data', number=200000, globals=globals()))
print("set membership:", timeit.timeit('9999 in s_data', number=200000, globals=globals()))


### 3) Разбор кода
Сравниваются размеры контейнеров и скорость операции `in`. У set membership обычно значительно быстрее на больших коллекциях.

### 4) Производительность и память
`set` потребляет больше памяти, но выигрывает по поиску. `tuple` часто выгоден как read-only структура и hashable-ключ (если элементы hashable).

### 5) Вопросы с собеседований

1. Почему `set` быстрее для membership, но не хранит порядок как list?
2. Когда tuple лучше list в API-дизайне?


### 6) Практические мини-задачи

1. Выберите оптимальный контейнер для feature whitelist и обоснуйте.
2. Перепишите код, где membership в list на горячем пути, используя set.


### 7) Edge cases

- `set` не поддерживает индексирование.
- Малые структуры могут не показать выигрыш set из-за константных факторов.


## Iterator vs Iterable

### 1) Теория

**Iterable** — объект, который может вернуть итератор (`__iter__`).  
**Iterator** — объект с `__next__`, который выдаёт элементы и помнит состояние обхода.


In [ ]:
nums = [1, 2, 3]
it = iter(nums)
print(next(it), next(it), next(it))

class Countdown:
    def __init__(self, start):
        self.start = start
    def __iter__(self):
        n = self.start
        while n > 0:
            yield n
            n -= 1

print(list(Countdown(5)))


### 3) Разбор кода
Список — iterable, `iter(nums)` возвращает iterator. `next` двигает внутреннее состояние. Класс `Countdown` возвращает генератор-итератор.

### 4) Производительность и память
Итераторы ленивы и экономят память, что критично для потоковой обработки данных в ML-pipeline.

### 5) Вопросы с собеседований

1. Почему iterator нельзя “перемотать” без создания нового?
2. В чём польза протокола итерации для больших датасетов?


### 6) Практические мини-задачи

1. Напишите iterable-обёртку над чтением строк из файла батчами.
2. Покажите, что один и тот же iterator истощается.


### 7) Edge cases

- Итератор после исчерпания всегда кидает `StopIteration`.
- Ошибки часто возникают при повторном использовании уже истощённого iterator.


## Generators

### 1) Теория

Генератор — способ создавать итератор через `yield`, сохраняя состояние кадра функции между вызовами.  
Даёт ленивые вычисления, backpressure-friendly обработку потоков.


In [ ]:
def squares(n):
    for i in range(n):
        yield i * i

g = squares(5)
print(next(g))
print(list(g))  # оставшиеся элементы

big_gen = (i for i in range(10_000_000))
print(type(big_gen))


### 3) Разбор кода
`yield` превращает функцию в генератор. После `next(g)` генератор продолжает с места остановки. Генераторные выражения удобны для потоковой обработки.

### 4) Производительность и память
Генераторы минимизируют пиковое потребление RAM, но одноразовые и иногда медленнее в CPU из-за накладных расходов итерации.

### 5) Вопросы с собеседований

1. Когда генератор лучше списка в ETL-задачах?
2. Как обрабатывать исключения внутри генератора?


### 6) Практические мини-задачи

1. Напишите генератор, который читает лог-файл и фильтрует строки по паттерну.
2. Добавьте пагинацию батчами через `yield` списков фиксированного размера.


### 7) Edge cases

- Генератор нельзя напрямую сериализовать как данные.
- Повторная итерация требует создать новый генератор.


## List comprehension vs generator expression

### 1) Теория

`[f(x) for x in data]` сразу создаёт список в памяти.  
`(f(x) for x in data)` создаёт генератор с ленивым вычислением.


In [ ]:
data = range(1_000_000)
list_comp_time = timeit.timeit('[x * 2 for x in data]', number=5, globals=globals())
gen_expr_time = timeit.timeit('sum(x * 2 for x in data)', number=5, globals=globals())

print("list comprehension time:", round(list_comp_time, 4))
print("generator expression with sum time:", round(gen_expr_time, 4))

lst = [x * 2 for x in range(5)]
gen = (x * 2 for x in range(5))
print(lst, list(gen))


### 3) Разбор кода
List comprehension материализует весь результат сразу. Generator expression вычисляет по мере потребления. Время зависит от паттерна использования и оптимизаций C-функций (`sum`).

### 4) Производительность и память
Для больших объёмов данных generator expression экономит RAM. Если результат нужен многократно/рандомно по индексу — лучше list.

### 5) Вопросы с собеседований

1. Почему list comprehension обычно быстрее “чисто вычислительно” при полной материализации?
2. Когда generator expression критически снижает memory footprint?


### 6) Практические мини-задачи

1. Измерьте RAM-профиль для list vs generator на 10 млн элементов.
2. Замените временный list в пайплайне на chain генераторов.


### 7) Edge cases

- Генератор “исчезает” после одного прохода.
- Ошибки отложенных вычислений проявляются позже (в момент итерации).


## Big-O сложность частых операций

### 1) Теория

Нужно знать не только асимптотику, но и **константы**, locality CPU cache и распределение входных данных.

Быстрый справочник:
- `list.append` — амортизированно O(1)
- `list.pop()` с конца — O(1), `pop(0)` — O(n)
- `x in list` — O(n)
- `x in set/dict` — среднее O(1)
- `dict.get/set/del` — среднее O(1)
- сортировка Timsort — O(n log n)


In [ ]:
from collections import deque

n = 100_000
lst = list(range(n))
dq = deque(range(n))

print("list pop(0):", timeit.timeit('lst.pop(0); lst.insert(0, -1)', number=2000, globals=globals()))
print("deque popleft/appendleft:", timeit.timeit('dq.popleft(); dq.appendleft(-1)', number=2000, globals=globals()))


### 3) Разбор кода
Сравнивается удаление из начала list и deque. Для очередей deque обычно гораздо эффективнее.

### 4) Производительность и память
Выбор структуры данных — ключ к latency/throughput. Асимптотика + профиль реальной нагрузки важнее “микрооптимизаций синтаксиса”.

### 5) Вопросы с собеседований

1. Почему `pop(0)` у list дорогой?
2. Для каких workloads deque объективно лучше?


### 6) Практические мини-задачи

1. Замените очередь на list в legacy-коде на deque и измерьте ускорение.
2. Составьте таблицу операций и сложностей для используемых вами структур.


### 7) Edge cases

- На малых n list может быть не хуже из-за CPU cache и overhead deque.
- Плохие бенчмарки (без прогрева/шумов) дают неверные выводы.


## GIL: концепция и последствия

### 1) Теория

GIL (Global Interpreter Lock) в CPython разрешает выполнение байткода Python только одному потоку одновременно.  

Следствия:
- CPU-bound задачи не масштабируются по потокам внутри одного процесса CPython.
- I/O-bound задачи хорошо работают с threads, потому что GIL освобождается на I/O.
- Для CPU-bound обычно используют `multiprocessing`, C/C++ расширения, NumPy (часто освобождает GIL), или распределённые вычисления.


In [ ]:
import threading

counter = 0
lock = threading.Lock()

def work(n=100_000):
    global counter
    for _ in range(n):
        with lock:
            counter += 1

threads = [threading.Thread(target=work) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("counter:", counter)


### 3) Разбор кода
Пример с lock показывает корректную синхронизацию shared state. Даже с потоками CPU-bound Python-код не получает линейного ускорения из-за GIL.

### 4) Производительность и память
Для high-throughput ML сервисов: используйте процессы для CPU-heavy части, асинхронность/пулы потоков для I/O (сеть, диски, БД).

### 5) Вопросы с собеседований

1. Почему многопоточность в CPython может не ускорить CPU-bound код?
2. Когда лучше `threading`, когда `multiprocessing`, когда `asyncio`?


### 6) Практические мини-задачи

1. Сравните время CPU-bound функции в single-thread, multi-thread и multi-process.
2. Спроектируйте inference pipeline, где preprocessing CPU-bound, а запросы в feature-store I/O-bound.


### 7) Edge cases

- Race conditions остаются возможными несмотря на GIL (операции не всегда атомарны на уровне бизнес-логики).
- У альтернативных интерпретаторов (PyPy/Jython/IronPython) поведение отличается.



## Рекомендации по подготовке к интервью

- Уметь объяснить **почему** выбранная структура данных корректна по сложности и памяти.
- Показывать практику: `timeit`, `sys.getsizeof`, профилирование (`cProfile`, `line_profiler`, `memory_profiler`).
- Разбирать edge cases и trade-offs, а не только “правильный ответ”.
